# Asistente “Helpdesk de alumnos” con herramientas y confirmación

### Objetivo

Construir un asistente de consola que responda dudas de alumnos, con:

- agno.Agent + OpenAIChat

- una tool propia (p. ej. calcular_nota_final)

- una tool externa (simulada) que requiera confirmación antes de ejecutarse

- manejo de .env con dotenv para claves/API

### Requisitos previos

- Python 3.10+

- VSCode con extensión Python

- Paquetes: python-dotenv, agno, openinference-instrumentation-agno (no usar aún), httpx (solo si lo prefieres para la tool externa)

### Objetivos:

#### Objetivo 1

- Crear entorno virtual y proyecto helpdesk/.

- Configurar .env con OPENAI_API_KEY.

- Crear un Agent mínimo con OpenAIChat(id="gpt-4o-mini", temperature=0) e instrucción: “Eres un asistente de helpdesk académico”.

- Implementar CLI simple: el usuario escribe una pregunta, el agente responde (usa agent.print_response con stream=True).

In [ ]:
# Objetivo 1: Agente mínimo con OpenAIChat y CLI simple
import os
from dotenv import load_dotenv
from agno import Agent
from agno.models.openai import OpenAIChat

# Cargar la clave de API desde .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Crear el agente básico
agent = Agent(
    model=OpenAIChat(
        id="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY
    ),
    instructions="Eres un asistente de helpdesk académico."
)

# CLI simple
print("Asistente Helpdesk - Escribe 'salir' para terminar")
print("-" * 50)

while True:
    pregunta = input("\nPregunta: ")
    if pregunta.lower() == "salir":
        print("¡Hasta luego!")
        break
    agent.print_response(pregunta, stream=True)

#### Objetivo 2

- Implementar una función calcular_nota_final(partial, exam, extra) que devuelva la media ponderada (p. ej. 40%, 50%, 10%).

- Registrar la función como tool del Agent para que pueda llamarla cuando el usuario pregunte por su nota.

- Validar inputs (tipos, rangos 0–10) y devolver errores legibles.


In [ ]:
# Objetivo 2: Tool calcular_nota_final con validaciones
import os
from dotenv import load_dotenv
from agno import Agent
from agno.models.openai import OpenAIChat

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Definir la tool calcular_nota_final
def calcular_nota_final(partial: float, exam: float, extra: float) -> str:
    """
    Calcula la nota final ponderada de un alumno.
    
    Args:
        partial: Nota de evaluación parcial (0-10)
        exam: Nota del examen final (0-10)
        extra: Nota de trabajos extra (0-10)
    
    Returns:
        str: Mensaje con la nota final calculada
    """
    # Validar tipos
    try:
        partial = float(partial)
        exam = float(exam)
        extra = float(extra)
    except (ValueError, TypeError):
        return "Error: Todas las notas deben ser números válidos."
    
    # Validar rangos
    if not (0 <= partial <= 10):
        return "Error: La nota parcial debe estar entre 0 y 10."
    if not (0 <= exam <= 10):
        return "Error: La nota del examen debe estar entre 0 y 10."
    if not (0 <= extra <= 10):
        return "Error: La nota extra debe estar entre 0 y 10."
    
    # Calcular nota final con ponderación: 40% parcial, 50% examen, 10% extra
    nota_final = (partial * 0.4) + (exam * 0.5) + (extra * 0.1)
    
    return f"Nota final calculada: {nota_final:.2f}/10 (Parcial: {partial}, Examen: {exam}, Extra: {extra})"

# Crear agente con la tool
agent_con_tools = Agent(
    model=OpenAIChat(
        id="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY
    ),
    instructions="Eres un asistente de helpdesk académico. Puedes calcular notas finales cuando te las pidan.",
    tools=[calcular_nota_final],
    show_tool_calls=True
)

# Probar la tool
print("Agente con tool calcular_nota_final configurado")
print("\nPrueba: '¿Cuál es mi nota final si tengo 7 en parcial, 8 en examen y 9 en extra?'")
agent_con_tools.print_response(
    "¿Cuál es mi nota final si tengo 7 en parcial, 8 en examen y 9 en extra?",
    stream=True
)

#### Objetivo 3

- Crear una tool enviar_email_recordatorio(correo, asunto, cuerpo) que simule envío (no real): escribe un log en ./outbox/.

- Añadir un hook de confirmación (patrón similar a tu ejemplo con Prompt.ask) que, antes de ejecutar, pregunte “¿Quieres que proceda? (s/n)”.

- Si el usuario no confirma, cancelar educadamente sin ejecutar.

In [ ]:
# Objetivo 3: Tool enviar_email_recordatorio con confirmación
import os
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from agno import Agent, RunResponse
from agno.models.openai import OpenAIChat
from agno.tools.function import Function

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Crear directorio outbox si no existe
outbox_dir = Path("./outbox")
outbox_dir.mkdir(exist_ok=True)

# Variable global para controlar confirmación
confirmacion_usuario = None

def enviar_email_recordatorio(correo: str, asunto: str, cuerpo: str) -> str:
    """
    Simula el envío de un email recordatorio escribiendo un log en ./outbox/.
    Requiere confirmación del usuario antes de ejecutarse.
    
    Args:
        correo: Dirección de correo del destinatario
        asunto: Asunto del email
        cuerpo: Contenido del email
    
    Returns:
        str: Mensaje de confirmación o cancelación
    """
    global confirmacion_usuario
    
    # Solicitar confirmación
    print(f"\n{'='*60}")
    print("CONFIRMACIÓN REQUERIDA")
    print(f"{'='*60}")
    print(f"Correo: {correo}")
    print(f"Asunto: {asunto}")
    print(f"Cuerpo: {cuerpo}")
    print(f"{'='*60}")
    
    confirmacion = input("¿Quieres que proceda con el envío? (s/n): ").lower()
    
    if confirmacion != 's':
        return "Envío cancelado por el usuario. No se ha enviado ningún email."
    
    # Simular envío: escribir log en outbox
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"email_{timestamp}.txt"
    filepath = outbox_dir / filename
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f"Timestamp: {datetime.now().isoformat()}\n")
        f.write(f"Para: {correo}\n")
        f.write(f"Asunto: {asunto}\n")
        f.write(f"{'='*60}\n")
        f.write(f"Cuerpo:\n{cuerpo}\n")
    
    return f"Email enviado exitosamente a {correo}. Log guardado en {filepath}"

# Crear agente con ambas tools
agent_completo = Agent(
    model=OpenAIChat(
        id="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY
    ),
    instructions="Eres un asistente de helpdesk académico. Puedes calcular notas y enviar emails recordatorios.",
    tools=[calcular_nota_final, enviar_email_recordatorio],
    show_tool_calls=True
)

print("Agente con tool enviar_email_recordatorio configurado")
print("\nPrueba: 'Envía un recordatorio a alumno@university.edu sobre la entrega del proyecto'")
agent_completo.print_response(
    "Envía un recordatorio a alumno@university.edu sobre la entrega del proyecto final que vence el viernes",
    stream=True
)

#### Objetivo 4

- Implementar una tool buscar_politica_eval(asignatura) que simule consultar una política (devuelve un dict desde un JSON local o un endpoint público inofensivo).

- Integrarla en el agente: si el usuario pregunta “¿cómo se evalúa {X}?”, usar la tool.

In [ ]:
# Objetivo 4: Tool buscar_politica_eval
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from agno import Agent
from agno.models.openai import OpenAIChat

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Crear archivo JSON con políticas de evaluación
politicas_file = Path("./politicas_evaluacion.json")

if not politicas_file.exists():
    politicas_data = {
        "Matemáticas": {
            "evaluacion_continua": "40%",
            "examen_parcial": "20%",
            "examen_final": "30%",
            "trabajos_practicos": "10%",
            "requisitos": "Mínimo 5/10 en examen final para aprobar"
        },
        "Programación": {
            "evaluacion_continua": "30%",
            "proyectos": "40%",
            "examen_final": "20%",
            "participacion": "10%",
            "requisitos": "Todos los proyectos deben estar entregados"
        },
        "Inteligencia Artificial": {
            "evaluacion_continua": "25%",
            "proyectos": "35%",
            "examen_teorico": "25%",
            "presentaciones": "15%",
            "requisitos": "Asistencia mínima 80%"
        },
        "Bases de Datos": {
            "evaluacion_continua": "35%",
            "practicas_laboratorio": "25%",
            "examen_final": "30%",
            "proyecto_final": "10%",
            "requisitos": "Aprobar todas las prácticas de laboratorio"
        }
    }
    
    with open(politicas_file, 'w', encoding='utf-8') as f:
        json.dump(politicas_data, f, ensure_ascii=False, indent=2)
    
    print(f"Archivo de políticas creado: {politicas_file}")

def buscar_politica_eval(asignatura: str) -> str:
    """
    Busca y retorna la política de evaluación de una asignatura.
    
    Args:
        asignatura: Nombre de la asignatura a consultar
    
    Returns:
        str: Información sobre la política de evaluación de la asignatura
    """
    try:
        with open(politicas_file, 'r', encoding='utf-8') as f:
            politicas = json.load(f)
        
        # Buscar la asignatura (case-insensitive)
        asignatura_key = None
        for key in politicas.keys():
            if key.lower() == asignatura.lower():
                asignatura_key = key
                break
        
        if asignatura_key:
            politica = politicas[asignatura_key]
            resultado = f"Política de evaluación para {asignatura_key}:\n\n"
            
            for criterio, valor in politica.items():
                criterio_formato = criterio.replace('_', ' ').title()
                resultado += f"- {criterio_formato}: {valor}\n"
            
            return resultado
        else:
            asignaturas_disponibles = ", ".join(politicas.keys())
            return f"No se encontró la asignatura '{asignatura}'. Asignaturas disponibles: {asignaturas_disponibles}"
    
    except Exception as e:
        return f"Error al consultar la política de evaluación: {str(e)}"

# Crear agente con todas las tools
agent_final = Agent(
    model=OpenAIChat(
        id="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY
    ),
    instructions="""Eres un asistente de helpdesk académico. Puedes:
    1. Calcular notas finales de los alumnos
    2. Enviar emails recordatorios (con confirmación)
    3. Consultar políticas de evaluación de asignaturas""",
    tools=[calcular_nota_final, enviar_email_recordatorio, buscar_politica_eval],
    show_tool_calls=True
)

print("Agente con tool buscar_politica_eval configurado")
print("\nPrueba: '¿Cómo se evalúa Programación?'")
agent_final.print_response("¿Cómo se evalúa Programación?", stream=True)

#### Objetivo 5

- Escribir un README con instrucciones de ejecución.

- Añadir .env.example.

- Añadir requirements.txt

In [ ]:
# Objetivo 5: Crear archivos de documentación y configuración

# 1. Crear README.md
readme_content = """# Asistente Helpdesk Académico

## Descripción
Asistente de consola inteligente para responder dudas de alumnos, utilizando **agno** y **OpenAI GPT-4o-mini**.

## Características
- 🤖 Agente conversacional con CLI interactiva
- 📊 Cálculo de notas finales ponderadas
- 📧 Envío simulado de emails recordatorios (con confirmación)
- 📋 Consulta de políticas de evaluación de asignaturas

## Requisitos
- Python 3.10+
- Cuenta de OpenAI con API key

## Instalación

1. **Clonar o descargar el proyecto**

2. **Crear entorno virtual**
   ```bash
   python -m venv venv
   ```

3. **Activar el entorno virtual**
   - Windows:
     ```bash
     venv\\Scripts\\activate
     ```
   - Linux/Mac:
     ```bash
     source venv/bin/activate
     ```

4. **Instalar dependencias**
   ```bash
   pip install -r requirements.txt
   ```

5. **Configurar variables de entorno**
   - Copiar `.env.example` a `.env`
   - Añadir tu API key de OpenAI:
     ```
     OPENAI_API_KEY=tu-api-key-aqui
     ```

## Uso

### Ejecutar en Jupyter Notebook
Ejecuta las celdas del notebook `1. Asistente Helpdesk.ipynb` en orden.

### Funcionalidades

#### 1. Calcular Nota Final
Pregunta: "¿Cuál es mi nota final si tengo 7 en parcial, 8 en examen y 9 en extra?"

#### 2. Enviar Email Recordatorio
Pregunta: "Envía un recordatorio a alumno@university.edu sobre la entrega del proyecto"
- El sistema solicitará confirmación antes de enviar

#### 3. Consultar Política de Evaluación
Pregunta: "¿Cómo se evalúa Programación?"

## Estructura del Proyecto
```
helpdesk/
├── 1. Asistente Helpdesk.ipynb  # Notebook principal
├── README.md                     # Este archivo
├── requirements.txt              # Dependencias
├── .env                          # Configuración (no incluido en repo)
├── .env.example                  # Plantilla de configuración
├── politicas_evaluacion.json    # Base de datos de políticas
└── outbox/                       # Logs de emails enviados
```

## Asignaturas Disponibles
- Matemáticas
- Programación
- Inteligencia Artificial
- Bases de Datos

## Notas Técnicas
- **Ponderación de notas**: 40% parcial, 50% examen, 10% extra
- **Emails**: Simulados, se guardan en `./outbox/`
- **Políticas**: Editables en `politicas_evaluacion.json`

## Autor
Proyecto educativo - Módulo 6: Agentes Autónomos de IA
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✓ README.md creado")

# 2. Crear .env.example
env_example = """# Configuración de API Keys

# OpenAI API Key
# Obtén tu clave en: https://platform.openai.com/api-keys
OPENAI_API_KEY=your-openai-api-key-here
"""

with open(".env.example", "w", encoding="utf-8") as f:
    f.write(env_example)

print("✓ .env.example creado")

# 3. Crear requirements.txt
requirements = """# Dependencias del Asistente Helpdesk

# Framework de agentes
agno>=0.1.0

# Cliente OpenAI
openai>=1.0.0

# Gestión de variables de entorno
python-dotenv>=1.0.0

# Opcional: para instrumentación y observabilidad
# openinference-instrumentation-agno>=0.1.0

# Opcional: cliente HTTP para APIs externas
httpx>=0.25.0
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("✓ requirements.txt creado")

print("\n" + "="*60)
print("ARCHIVOS DE DOCUMENTACIÓN CREADOS EXITOSAMENTE")
print("="*60)
print("\nArchivos creados:")
print("  ✓ README.md")
print("  ✓ .env.example")
print("  ✓ requirements.txt")
print("\nPróximos pasos:")
print("  1. Copia .env.example a .env")
print("  2. Añade tu OPENAI_API_KEY en .env")
print("  3. Ejecuta: pip install -r requirements.txt")
print("  4. ¡Prueba el asistente!")
print("="*60)